# Task 1.2: Key Assumptions

**Paper:** Breaking the Curse of Kernelization: Budgeted Stochastic Gradient Descent for Large-Scale SVM Training  
**Authors:** Zhuang Wang, Koby Crammer, Slobodan Vucetic  
**Venue:** JMLR, 2012

---

The following three assumptions are specific to how the BSGD framework is designed and why it works. These are not general machine learning assumptions like "the data is i.i.d." but rather structural requirements embedded in the paper's algorithm, proofs, and budget maintenance strategies.

---

## Assumption 1: Bounded Kernel Feature Norm

**Assumption:** The method assumes that every data point has a bounded norm in the kernel-induced feature space, specifically $\|\Phi(x_t)\| \leq 1$ for all training examples. For translation-invariant kernels like the Gaussian RBF, this holds automatically because $k(x, x) = 1$. For other kernels such as polynomial kernels this is not guaranteed unless the data is normalised beforehand.

**Why the method needs it:** This bounded norm is critical for the convergence analysis. All three theorems in the paper (Theorems 1, 2, and 3 in Section 4) state this as an explicit precondition. The subgradient of the hinge loss has norm at most $\|\Phi(x_t)\|$, so if features are unbounded the gradient steps can be arbitrarily large, making the regret bounds meaningless. The Pegasos learning rate $\eta_t = 1/(\lambda t)$ is also calibrated under this assumption, and if feature norms were larger the step size would overshoot.

**Violation scenario:** Consider applying BSGD with a polynomial kernel $k(x, x') = (1 + x^T x')^d$ on raw image pixel data where pixel values range from 0 to 255. Without normalisation, $k(x, x)$ can be enormous, meaning $\|\Phi(x)\| \gg 1$. In this setting, the convergence guarantees would not hold and the algorithm could exhibit erratic coefficient growth or oscillating accuracy.

**Paper reference:** The assumption appears explicitly at the beginning of Theorems 1, 2, and 3 (Section 4). The learning rate table (Table 2) and the subgradient bounds in Algorithm 1 are all derived under this condition.

---

## Assumption 2: Budget Representability

**Assumption:** The method assumes that the optimal decision boundary can be adequately approximated using at most $B$ support vectors. More precisely, the averaged weight degradation $\bar{E} = \frac{1}{N}\sum_{t=1}^{N} \|\Delta_t\|$ must remain small throughout training for the budgeted solution to be close to the optimal unbounded SVM solution.

**Why the method needs it:** The entire BSGD framework relies on the idea that a fixed-size set of support vectors can capture the essential structure of the classifier. Theorems 1 and 2 make this explicit: the gap between the BSGD solution and the optimal solution grows proportionally with $\bar{E}$. If the true classifier genuinely requires thousands of support vectors to represent its boundary (because the boundary is very complex) then compressing it into $B$ vectors will introduce large errors at every budget maintenance step, and these errors accumulate. The merging strategy further assumes that pairs of nearby support vectors exist and can be meaningfully combined.

**Violation scenario:** The paper itself demonstrates this violation on the Covertype dataset, which has 54 features and 7 classes with an intricate decision boundary. With $B=100$, BPegasos+merge achieves only about 65.6% accuracy while unbounded Pegasos reaches 80.3% (Table 5, Figure 2b). The budget simply cannot capture enough structure, and $\bar{E}$ stays high throughout training.

**Paper reference:** Section 4 (Theorems 1 through 3, where the $\bar{E}$ term appears), Table 6 (showing how $\bar{E}$ decreases as $B$ increases), and Figure 2b (showing the accuracy-vs-budget trade-off on Covertype).

---

## Assumption 3: Locality of Support Vectors for Merging

**Assumption:** The merging budget maintenance strategy assumes that when two support vectors are close in the kernel feature space (i.e., $k(x_m, x_n)$ is high), their contributions to the decision function are correlated and can be replaced by a single SV at a weighted midpoint without significantly degrading the classifier.

**Why the method needs it:** The merging operation (Section 6.3) constructs a virtual support vector $z = h \cdot x_m + (1-h) \cdot x_n$ and assigns it the combined coefficient $\alpha_z = \alpha_m + \alpha_n$. This works well when $k(x_m, x_n) \approx 1$, because then $k(z, \cdot) \approx k(x_m, \cdot) \approx k(x_n, \cdot)$, meaning the merged SV behaves almost identically to either original. But if the kernel varies rapidly in the region between $x_m$ and $x_n$ (for example, because they lie on opposite sides of a class boundary despite being close in input space), the merged SV will misrepresent both. The weight degradation for merging (Equations 15 through 17) is only small when the two SVs are genuinely "similar" in terms of their kernel representation.

**Violation scenario:** Consider a dataset with a very fine-grained decision boundary, such as a high-resolution checkerboard with thin stripes. Two SVs might be geometrically close ($\|x_m - x_n\|$ is small) but support different local parts of the boundary. With a narrow kernel width ($\sigma$ is very small), even this small gap leads to $k(x_m, x_n) \approx 0$, meaning the kernel treats them as very different. Merging them would place $z$ in a no-man's-land that does not represent either original SV's role in the classifier.

**Paper reference:** Section 6.3 (merging strategy, Equations 15 through 17), Table 6 (comparing weight degradation $\bar{E}$ across strategies), and Section 7.12 with Figure 7 (showing that merged SVs on USPS appear as "blurred" averages of training digits, which is a visual consequence of this locality assumption).